In [33]:
#Task 1：Steam 資料集統計（總量、唯一使用者、唯一遊戲）
import gzip
import json
import random
from collections import defaultdict

# 定義讀取函數
def parse(path):
    g = gzip.open(path, 'r')
    for l in g:
        yield eval(l)

# 讀取資料
dataset = []
path = "C:\\大二上\\coursera\\python p1期末作業\\australian_user_reviews.json.gz"

for d in parse(path):
    # 在 Steam 資料中，每一行可能包含多個 reviews
    user_id = d.get('user_id')
    for review in d.get('reviews', []):
        review['user_id'] = user_id
        dataset.append(review)

print(f"總共{len(dataset)} ")

print(dataset[0])

總共59305 
{'funny': '', 'posted': 'Posted November 5, 2011.', 'last_edited': '', 'item_id': '1250', 'helpful': 'No ratings yet', 'recommend': True, 'review': 'Simple yet with great replayability. In my opinion does "zombie" hordes and team work better than left 4 dead plus has a global leveling system. Alot of down to earth "zombie" splattering fun for the whole family. Amazed this sort of FPS is so rare.', 'user_id': '76561197970982479'}


In [34]:

import json
import random


# 1. 為了保持結果一致，設定隨機種子
random.seed(42)
random.shuffle(dataset)

# 2. 分割資料 (80% 訓練, 20% 測試)
train_size = int(len(dataset) * 0.8)
trainingSet = dataset[:train_size]
testSet = dataset[train_size:]

print(f"訓練集大小: {len(trainingSet)}")
print(f"測試集大小: {len(testSet)}")

# 3. 提取基礎統計量 (僅基於訓練集)
users = set()
items = set()
recommend_count = 0

for d in trainingSet:
    users.add(d['user_id'])
    items.add(d['item_id'])
    if d['recommend'] == True:
        recommend_count += 1

avg_recommend_rate = recommend_count / len(trainingSet)

print("\n訓練集大概統計的結果是")
print(f"1.Fraction of Recommended: {avg_recommend_rate:.4f}")
print(f"2.Unique Users: {len(users)}")
print(f"3.Unique Items: {len(items)}")

訓練集大小: 47444
測試集大小: 11861

訓練集大概統計的結果是
1.Fraction of Recommended: 0.8850
2.Unique Users: 22542
3.Unique Items: 3333


資料集名稱：Steam Video Game User Reviews (UCSD 資料庫) 。
包含 59,305 條真實玩家對 Steam 遊戲的評價。
關鍵欄位：user_id、item_id、recommend及review。
採取課程建議的 80/20 分割策略 ，確保模型評估的公正性，並刪除原始完整數據集以防誤用 。

In [35]:
#分類任務 (Task 2) — 預測玩家是否推薦
from sklearn import linear_model
import numpy as np

# 定義特徵函數
# 使用 [1 (偏移量), 評論文字長度] 當特徵 
def feature(datum):
    feat = [1]
    feat.append(len(datum.get('review', "")))
    return feat

# 準備訓練數據
X_train = [feature(d) for d in trainingSet]
y_train = [d['recommend'] for d in trainingSet]

# 訓練邏輯回歸模型
clf = linear_model.LogisticRegression(class_weight='balanced') # 加入 Balance權重處理不平衡問題
clf.fit(X_train, y_train)

# 在測試集上進行預測
X_test = [feature(d) for d in testSet]
y_test = [d['recommend'] for d in testSet]
predictions = clf.predict(X_test)

# 計算準確率
correct = predictions == y_test
accuracy = sum(correct) / len(correct)

# 計算BER
TP = sum([(p and l) for (p, l) in zip(predictions, y_test)])
FP = sum([(p and not l) for (p, l) in zip(predictions, y_test)])
TN = sum([(not p and not l) for (p, l) in zip(predictions, y_test)])
FN = sum([(not p and l) for (p, l) in zip(predictions, y_test)])

TPR = TP / (TP + FN) # 真陽性率
TNR = TN / (TN + FP) # 真陰性率
ber = 1 - 0.5 * (TPR + TNR)

print(f"分類模型準確率 (Accuracy): {accuracy:.4f}")
print(f"平衡錯誤率 (BER): {ber:.4f}")

分類模型準確率 (Accuracy): 0.7322
平衡錯誤率 (BER): 0.4563


In [10]:
pip install nltk


In [36]:
#回歸任務 (Task 3) — 情感分析與字詞特徵
import string
from collections import defaultdict
from sklearn import linear_model
from nltk.stem.porter import PorterStemmer

# 準備文字處理工具
punctuation = set(string.punctuation)
stemmer = PorterStemmer()

# 統計詞頻 (僅取訓練集的前 1000 筆))
wordCount = defaultdict(int)
for d in trainingSet[:10000]: # 取部分資料進行詞頻分析
    r = ''.join([c for c in d['review'].lower() if not c in punctuation])
    for w in r.split():
        wordCount[w] += 1

# 選最熱門的 100 個單字作為特徵
counts = [(wordCount[w], w) for w in wordCount]
counts.sort(reverse=True)
words = [x[1] for x in counts[:100]]
wordId = dict(zip(words, range(len(words))))
wordSet = set(words)

# 定義回歸特徵函數
def feature_reg(datum):
    feat = [0] * len(words)
    r = ''.join([c for c in datum['review'].lower() if not c in punctuation])
    for w in r.split():
        if w in wordSet:
            feat[wordId[w]] += 1
    feat.append(1) 
    return feat

# 建立訓練與測試向量
# y向量：將True轉為1,False轉為0
y_train_reg = [1 if d['recommend'] else 0 for d in trainingSet]
y_test_reg = [1 if d['recommend'] else 0 for d in testSet]

X_train_reg = [feature_reg(d) for d in trainingSet[:10000]]
y_train_reg = y_train_reg[:10000]

# 訓練Ridge回歸模型
model = linear_model.Ridge(alpha=1.0, fit_intercept=True)
model.fit(X_train_reg, y_train_reg)

# 預測與計算MSE
X_test_reg = [feature_reg(d) for d in testSet]
predictions = model.predict(X_test_reg)

def MSE(predictions, labels):
    differences = [(x-y)**2 for x,y in zip(predictions, labels)]
    return sum(differences) / len(differences)

mse_result = MSE(predictions, y_test_reg)
print(f"回歸模型MSE:{mse_result:.4f}")

回歸模型MSE:0.1108


In [37]:
#Task 4：推薦系統優化結果
import scipy.optimize
from collections import defaultdict
import numpy as np

# 建立推薦系統所需的字典數據結構 
reviewsPerUser = defaultdict(list)
reviewsPerItem = defaultdict(list)
for d in trainingSet:
    user, item = d['user_id'], d['item_id']
    reviewsPerUser[user].append(d)
    reviewsPerItem[item].append(d)
y_rec = [1 if d['recommend'] else 0 for d in trainingSet]
ratingMean = sum(y_rec) / len(y_rec)

# 算MSE
labels = y_rec
baseLineMSE = sum([(ratingMean - l)**2 for l in labels]) / len(labels)
print(f"1. 基準線平均推薦率: {ratingMean:.4f}")
print(f"2. 基準線 MSE: {baseLineMSE:.4f}")

# 定義優化所需的變數與函數
N = len(trainingSet)
nUsers = len(reviewsPerUser)
nItems = len(reviewsPerItem)
users = list(reviewsPerUser.keys())
items = list(reviewsPerItem.keys())

alpha = ratingMean
userBiases = defaultdict(float)
itemBiases = defaultdict(float)

def unpack(theta):
    global alpha
    global userBiases
    global itemBiases
    alpha = theta[0]
    userBiases = dict(zip(users, theta[1:nUsers+1]))
    itemBiases = dict(zip(items, theta[1+nUsers:]))

def cost(theta, labels, lamb):
    unpack(theta)
    cost = 0
    for d in trainingSet:
        u, i = d['user_id'], d['item_id']
        pred = alpha + userBiases.get(u, 0) + itemBiases.get(i, 0)
        cost += (pred - (1 if d['recommend'] else 0))**2
    
    cost /= len(trainingSet)
    for u in userBiases: cost += lamb * userBiases[u]**2
    for i in itemBiases: cost += lamb * itemBiases[i]**2
    return cost

def derivative(theta, labels, lamb):
    unpack(theta)
    dalpha = 0
    dUserBiases = defaultdict(float)
    dItemBiases = defaultdict(float)
    
    for d in trainingSet:
        u, i = d['user_id'], d['item_id']
        pred = alpha + userBiases.get(u, 0) + itemBiases.get(i, 0)
        diff = pred - (1 if d['recommend'] else 0)
        dalpha += 2/N * diff
        dUserBiases[u] += 2/N * diff
        dItemBiases[i] += 2/N * diff
        
    for u in users: dUserBiases[u] += 2 * lamb * userBiases[u]
    for i in items: dItemBiases[i] += 2 * lamb * itemBiases[i]
    
    dtheta = [dalpha] + [dUserBiases[u] for u in users] + [dItemBiases[i] for i in items]
    return np.array(dtheta)


initial_theta = [ratingMean] + [0.0]*nUsers + [0.0]*nItems
res = scipy.optimize.fmin_l_bfgs_b(cost, initial_theta, derivative, args=(y_rec, 1.0))
optimized_theta = res[0]
final_mse = cost(optimized_theta, y_rec, 0) # 計算不含正則化項的真實 MSE

print(f"3. 優化後的訓練集 MSE: {final_mse:.4f}")

1. 基準線平均推薦率: 0.8850
2. 基準線 MSE: 0.1018
3. 優化後的訓練集 MSE: 0.1017


In [30]:
# Testing^_^
# 從訓練集中選一個使用者和一個遊戲，看看模型預測他會推薦的機率
sample_d = trainingSet[0]
test_user = sample_d['user_id']
test_item = sample_d['item_id']

# 計算預測得分
prediction_score = alpha + userBiases.get(test_user, 0) + itemBiases.get(test_item, 0)

print(f"--- 報告情境測試結果 ---")
print(f"模擬使用者 ID: {test_user}")
print(f"模擬遊戲 ID: {test_item}")
print(f"實際是否推薦: {'大推' if sample_d['recommend'] else '不推薦'}")
print(f"模型預測得分: {prediction_score:.2f} (越接近 1 代表越可能推薦)")

--- 報告情境測試結果 ---
模擬使用者 ID: skyfii
模擬遊戲 ID: 440
實際是否推薦: 大推
模型預測得分: 0.89 (越接近 1 代表越可能推薦)
